# Workflow 3: Modelling and variable selection approaches

## Inputs

In [8]:
# Import libraries
import os
import pandas as pd
import exploration_modelling_functions as ef
import glob
from joblib import Parallel, delayed # Only for variable selection
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [9]:
# === Initialize Project Paths and Folders ===

# Define the project name and base working directory
project_name="dmdox_recalculation_13032026"
working_directory="C:/Users/Pheno/Documents/database_almondcv3/"

# Define the project and results directory
project_directory=os.path.join(working_directory, project_name)
results_directory=os.path.join(project_directory, f"Results_exploration_{project_name}")

# Create the necessary directories if they don't already exist
if not os.path.exists(project_directory):

    os.makedirs(project_directory)
if not os.path.exists(results_directory):

    os.makedirs(results_directory)

# Define the spectral data path
file_data_path= r"C:\Users\Pheno\Documents\database_almondcv3\RESULTADOS_PAPER\Datasets_curated\Kernel\all_results_joined.txt"

# Define the features/labels data path
file_labels_path=os.path.join(working_directory, r"Training_dataset_paper_hyper\LABELS_familia.txt")


## Data integration

In [10]:
# === Integrates spectral data and features/labels for data exploration ===

# Import spectral data (tab-delimited file, force column "ID" to be a categorical type)
df_all_results = pd.read_csv(file_data_path, delimiter='\t', dtype={'ID': 'category'})

# Extract all preprocessing methods (columns that start with "Mean" or "Median")
preprocessing_methods = df_all_results.filter(regex="^(Mean|Median)").columns.tolist()

# Import features/labels (tab-delimited file, "ID" also as categorical to ensure consistent merging)
df_labels = pd.read_csv(file_labels_path, delimiter='\t', dtype={'ID': 'category'})

# Merge spectral data with labels/features on "ID"
# - on='ID' → join by sample identifier
# - how='left' → keep all rows from df_all_results, add matching labels from df_labels
df_all_results = pd.merge(df_all_results, df_labels, on='ID', how='left')

# Display the merged DataFrame (uncomment the line below if you want to preview the data in the console)
# df_all_results

# Display the list of preprocessing methods (uncomment the line below to preview it)
# preprocessing_methods

C:\Users\Pheno\AppData\Local\Temp\ipykernel_21700\478439259.py:4: DtypeWarning: Columns (55) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all_results = pd.read_csv(file_data_path, delimiter='\t', dtype={'ID': 'category'})


## Data curation

In [11]:
# Remove samples
# Create an empty list of IDs to remove.
remove_ids = ["a"]
# Keep only the rows where the "ID" column is NOT in remove_ids
df_all_results = df_all_results[~df_all_results["ID"].isin(remove_ids)]

In [12]:
# Remove duplicates

# Count how many times each combination of "Band" and "ID" appears
duplicates = df_all_results.groupby(["Band", "ID"]).size().reset_index(name='count')

# Print only the rows where the same ("Band", "ID") pair appears more than once
print(duplicates[duplicates["count"] > 1])

# Remove duplicate rows, keeping only the first occurrence of each ("Band", "ID") pair
df_all_results = df_all_results.drop_duplicates(subset=["Band", "ID"])

        Band     ID  count
383      950  B1_94      2
1110     952  B1_94      2
1837     954  B1_94      2
2564     956  B1_94      2
3291     958  B1_94      2
...      ...    ...    ...
258468  1660  B1_94      2
259195  1662  B1_94      2
259922  1664  B1_94      2
260649  1666  B1_94      2
261376  1668  B1_94      2

[360 rows x 3 columns]


In [ ]:
# 1️⃣ Columns to keep fixed (metadata and traits of interest for modelling analyses)
id_vars_list = [
    'Session', 'Picture_name', 'ID', 'Band', 'Name', 'Farm', 
    'Fats', 'Moisture', 'Fiber', 'Protein', 'Sucrose', 
    'C_16', 'C_18', 'C_18_1', 'C_18_2', 'Cholesterol',
    'Campesterol', 'Stigmasterol', 'Betasitosterol', 
    'Delta7_Stigmasterol', 'Total_Sterols'
]

# 2️⃣ Spectral columns (preprocessed values we want to analyze/flatten)
spectral_cols = df_all_results.filter(regex="^(Mean|Median)").columns.tolist()

# 3️⃣ Convert the DataFrame from wide to long format
# - id_vars: columns to keep fixed (metadata, traits of interest for PCA)
# - value_vars: spectral metrics to flatten (Mean/Median columns)
# - var_name: new column with the name of the metric
# - value_name: new column with the corresponding value
df_long = pd.melt(
    df_all_results,
    id_vars=id_vars_list,      # fixed columns
    value_vars=spectral_cols,  # columns to flatten
    var_name='Metric',         # new column with metric names
    value_name='Value'         # new column with metric values
)

# Uncomment to display
# df_long

## PLS

In [ ]:
# Select traits
traits = [
    'Fats', 'Fiber'
]
# === Select preprocessing methods to include in the modeling ===

# Option 1: Include ALL available preprocessing methods
# preprocessing_selected = preprocessing_methods  

# Option 2: Select a subset of preprocessing methods (recommended for testing/comparison)
preprocessing_selected = ["Mean_SG1_W15_P2", "Median_SG1_W15_P2"]

# Option 3: Display all available preprocessing methods (uncomment to check the list)
# preprocessing_methods

In [ ]:
# =============================================================================
# General Description:
# --------------------
# This function trains and validates Partial Least Squares Regression (PLSR) models 
# using spectral data in long format. It allows to:
#   - Evaluate different preprocessing methods (RAW, SNV, MSC, SG, etc.).
#   - Test different numbers of PLS components.
#   - Perform repeated cross-validation (Repeated KFold).
#   - Optionally perform an external train/test split using Kennard-Stone. 
#   - Compute performance metrics such as RMSE and R² for train, test, and external sets.
#   - Save results into summary tables and generate boxplots/line plots.
#
# In short, it is used to evaluate the performance of PLS models depending on:
#   - the number of components,
#   - the preprocessing method applied,
#   - and the target trait to predict.
#
# =============================================================================
# Arguments:
#
# df_long : pd.DataFrame
#     Long-format DataFrame with columns: sample ID, Band, Value, Metric, and traits.
#
# traits : list
#     List of trait column names (dependent variables) to be predicted.
#
# preprocessed_metrics : list
#     List of preprocessing methods to include (e.g., RAW, SNV, MSC, ...).
#
# save_path : str
#     Directory path where the results (summaries and plots) will be saved.
#
# index_column : str, default 'ID'
#     Column name that uniquely identifies each sample.
#
# n_components_range : range, default range(1, 11)
#     Range of PLS components to evaluate.
#
# n_splits : int, default 7
#     Number of folds for cross-validation (KFold).
#
# n_repeats : int, default 50
#     Number of repetitions for repeated cross-validation (Repeated KFold).
#
# external_split : bool, default True
#     If True, an external split (train/external) is performed in addition to cross-validation.
#
# test_size : float, default 0.2
#     Proportion of data reserved as external set (e.g., 0.2 = 20%).
#
# random_state : int, default 10
#     Random seed to ensure reproducibility of splits.
#
# scaling : bool, default False
#     If True, standardizes spectral variables (mean=0, variance=1).
#
# =============================================================================

ef.pls_validation(df_long=df_long, traits=traits, preprocessed_metrics=preprocessing_selected, save_path=results_directory,
                index_column='ID', external_split=True, test_size=0.3, n_components_range=range(1,15),
                  n_splits=8, n_repeats=50, scaling=False)

## Variable selection through iPLS

In [ ]:
# 1️⃣ Columns to keep fixed (metadata and traits of interest for modelling analyses)
id_vars_list = [
    'Session', 'Picture_name', 'ID', 'Band', 'Name', 'Farm', 
    'Fats', 'Moisture', 'Fiber', 'Protein', 'Sucrose', 
    'C_16', 'C_18', 'C_18_1', 'C_18_2', 'Cholesterol',
    'Campesterol', 'Stigmasterol', 'Betasitosterol', 
    'Delta7_Stigmasterol', 'Total_Sterols'
]

# 2️⃣ Spectral columns (preprocessed values we want to analyze/flatten)
spectral_cols = df_all_results.filter(regex="^(Mean|Median)").columns.tolist()

# 3️⃣ Convert the DataFrame from wide to long format
# - id_vars: columns to keep fixed (metadata, traits of interest for PCA)
# - value_vars: spectral metrics to flatten (Mean/Median columns)
# - var_name: new column with the name of the metric
# - value_name: new column with the corresponding value
df_long = pd.melt(
    df_all_results,
    id_vars=id_vars_list,      # fixed columns
    value_vars=spectral_cols,  # columns to flatten
    var_name='Metric',         # new column with metric names
    value_name='Value'         # new column with metric values
)

# Uncomment to display
# df_long

In [ ]:
# =============================================================================
# Parameters for iPLS with Complexity Constraint
# =============================================================================

# Select traits (target variables) to predict
traits = [
    'Fats', "Fiber"
]

# Constraints to avoid unnecessary model complexity.
# If you don’t want to apply a constraint, set it to 1.
# This parameter defines how much the error must be reduced before allowing 
# an increase in the number of PLS components.
#
# Example:
#   - Current cross-validation error = 1 with n_components = 4
#   - If constraint = 0.85 → the error must be reduced by 15% (1 * 0.85 = 0.85) 
#     before accepting a model with more than 4 components.
#   - If constraint = 1 → no reduction is required, so higher components are always allowed.
constraints = [0.95, 1]

# Preprocessing method previously selected (only one at a time, based on prior PLS results)
preprocessed_method_selected = "Mean_SG1_W15_P2"

# Column that uniquely identifies each sample
index_column = 'ID'

# Number of parallel jobs for computation.
# Set to 1 if you don’t want parallelization (or if running on a single CPU).
n_parallel_jobs = 10

# Maximum number of PLS components to analyze
n_components = 3

# Cross-validation settings
n_splits = 8     # number of folds
n_repeats = 1   # number of repetitions

# External split settings
external_split = True
test_size = 0.3  # proportion of data reserved as external test set

# Interval sizes to test (number of bands per interval window in iPLS)
interval_sizes = [20]

# Random seed for reproducibility
random_state = 10

# Maximum number of interval combinations to test
max_number_interval_selected_tested = 3

full_mode=True
# full_mode : bool, default True
#     If True, perform full evaluation of all interval combinations, ignoring
#     temporary increases in RMSE. If False, stop adding intervals when the
#     prediction error no longer improves.
# =============================================================================

In [ ]:
# =============================================================================
# Interval Partial Least Squares (iPLS) with Complexity Constraint
# =============================================================================
# Description:
# ------------
# This implementation performs iPLS with a forward selection strategy.
# - It evaluates all possible interval combinations up to the maximum defined 
#   by `max_number_interval_selected_tested`.
# - The process continues until all possible intervals are tested if full_mode=True, 
#   providing a full/exhaustive evaluation. If full_mode=False, the selection stops 
#   when adding intervals does not improve prediction error.
# - It outputs all evaluated results, not just the best model.
# - Recommended to run on HPC (High-Performance Computing) environments if many 
#   combinations are to be tested, since the search space can be very large.
#
# The "complexity constraint" ensures that the number of components only increases 
# if the prediction error is sufficiently reduced (controlled by `constraints`).
# =============================================================================

def run_ipls(trait, constraint):
    print(f"Executing: {trait}, constraint: {constraint}", flush=True)
    ef.pls_interval_validation(
        df_long=df_long,
        trait=trait,
        preprocessed_metric=preprocessed_method_selected,
        save_path=results_directory,
        index_column=index_column,
        n_components=n_components,
        n_splits=n_splits,
        n_repeats=n_repeats,
        external_split=external_split,
        intervals_sizes=interval_sizes,
        test_size=test_size,
        random_state=random_state,
        max_number_interval_selected_tested=max_number_interval_selected_tested,
        constraint=constraint,
        full_mode= full_mode
    )

# Create all combinations of trait x constraint
combinations = [(trait, constraint) for trait in traits for constraint in constraints]

# Execute in parallel
Parallel(n_jobs=n_parallel_jobs)(
    delayed(run_ipls)(trait, constraint) for trait, constraint in combinations
)

In [ ]:
# =============================================================================
# Combine iPLS results
# =============================================================================

# Folder with iPLS result files
folder_path = os.path.join(results_directory, "iPLS_results")

# List to store DataFrames
df_list = []

# Read all .txt files in the folder
txt_files = glob.glob(os.path.join(folder_path, "*.txt"))

for file_path in txt_files:
    filename = os.path.basename(file_path)
    
    # Find the first trait that appears in the filename
    trait_found = next((trait for trait in traits if trait in filename), None)
    
    if trait_found:
        df = pd.read_csv(file_path, sep="\t")  # adjust separator if needed
        df["trait"] = trait_found
        df_list.append(df)
    else:
        print(f"⚠️ Trait not found in file: {filename}")

# Combine all DataFrames
combined_df = pd.concat(df_list, ignore_index=True)

# Save combined results to CSV
output_path = os.path.join(folder_path, "combined_results.csv")
combined_df.to_csv(output_path, index=False)

print(f"✅ Combined file saved at: {output_path}")



In [ ]:
# =============================================================================
# Selecting Top iPLS Models
# =============================================================================
# Description of the selection process:
# --------------------------------------
# This code performs a two-step ranking to select the best iPLS models for each trait:
#
# 1. First Filter (Top N1):
#    - Computes a combined distance metric based on RMSE_CV (val_rmse) and overfit ratio.
#      The distance is calculated as:
#          dist_rmse_overfit = sqrt(val_rmse_norm^2 + overfit_ratio_norm^2)
#    - The TOP_N1 models with the smallest distances are selected. This step prioritizes
#      models that have both low prediction error and low overfitting.
#
# 2. Second Filter (Top N2):
#    - Within the TOP_N1 subset, a second distance metric is computed considering
#      RMSE_CV and number of PLS components (n_components):
#          dist_rmse_components = sqrt(val_rmse_norm2^2 + n_components_norm2^2)
#    - This step favors models that achieve low error with fewer components (complexity constraint).
#    - The TOP_N2 models with the smallest distance are selected.
#
# 3. Final Ranking and Sorting:
#    - The TOP_N2 models are sorted by RMSE_CV and number of variables used.
#    - Each model is assigned a rank (1 = best).
#    - This rank represents the final selection of top models for that trait.
#
# 4. Visualizations:
#    - First plot: RMSE_CV vs. overfit ratio (shows first filtering step).
#    - Second plot: RMSE_CV vs. number of components (shows second filtering step with complexity consideration).
#    - Third plot: RMSE_CV vs. number of variables used (final selected models), highlighting
#      the complexity in terms of variable count.
#
# 5. Output:
#    - Saves individual CSVs for each trait with the selected top models.
#    - Saves plots for each trait showing the selection process.
#    - Saves a combined CSV for all traits.
#
# Summary:
# --------
# This procedure ensures that the selected iPLS models:
#   - Minimize cross-validation error (RMSE_CV)
#   - Avoid overfitting (overfit ratio close to 1)
#   - Use as few components as possible (complexity constraint)
#   - Use a manageable number of variables
# It provides a transparent, ranked selection for further analysis or reporting.
# =============================================================================

# ==== CONFIGURATION ====
TOP_N1 = 25  # First filter
TOP_N2 = 10  # Second filter

# Input CSV with all iPLS results
input_csv = output_path

# Output folder (where top models CSVs and plots will be saved)
output_dir = folder_path

# ==== PROCESS ====
all_selected = []

for trait in traits:
    # Load data for the trait
    df = pd.read_csv(input_csv)
    df = df[df["trait"] == trait].copy()

    # Derived features
    df["num_vars"] = df["interval_size"] * df["number_selected_interval"]
    df["val_rmse_norm"] = (df["val_rmse"] - df["val_rmse"].min()) / (df["val_rmse"].max() - df["val_rmse"].min())
    df["n_components_norm"] = (df["n_components"] - 1) / (15 - 1)
    df["num_vars_norm"] = ((df["num_vars"] - 5) / (360 - 5)).clip(0, 1)
    df["overfit_ratio_norm"] = (np.abs(df["overfit_ratio"] - 1) - np.abs(df["overfit_ratio"] - 1).min()) / \
                               (np.abs(df["overfit_ratio"] - 1).max() - np.abs(df["overfit_ratio"] - 1).min())
    df["overfit_ratio_norm"].fillna(0, inplace=True)

    # First selection (based on RMSE and overfit)
    df["dist_rmse_overfit"] = np.sqrt(df["val_rmse_norm"]**2 + df["overfit_ratio_norm"]**2)
    top_n1 = df.nsmallest(TOP_N1, "dist_rmse_overfit").copy()

    # Second selection (local scaling)
    top_n1["val_rmse_norm2"] = (top_n1["val_rmse"] - top_n1["val_rmse"].min()) / (top_n1["val_rmse"].max() - top_n1["val_rmse"].min())
    top_n1["n_components_norm2"] = (top_n1["n_components"] - top_n1["n_components"].min()) / (top_n1["n_components"].max() - top_n1["n_components"].min())
    top_n1["dist_rmse_components"] = np.sqrt(top_n1["val_rmse_norm2"]**2 + top_n1["n_components_norm2"]**2)
    top_n2 = top_n1.nsmallest(TOP_N2, "dist_rmse_components").copy()

    # Final sorting and ranking
    top_n2_sorted = top_n2.sort_values(by=["val_rmse", "num_vars"]).reset_index(drop=True)
    top_n2_sorted["rank"] = top_n2_sorted.index + 1
    top_n2_sorted["trait"] = trait
    all_selected.append(top_n2_sorted)

    # ==== PLOTS ====
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    sns.scatterplot(data=df, x="val_rmse", y="overfit_ratio", ax=axes[0], alpha=0.3)
    sns.scatterplot(data=top_n1, x="val_rmse", y="overfit_ratio", ax=axes[0], label=f"Top {TOP_N1}", color="red")
    axes[0].set_xlabel("RMSE_CV"); axes[0].set_ylabel("Overfit Ratio")

    sns.scatterplot(data=top_n1, x="val_rmse", y="n_components", ax=axes[1], label=f"Top {TOP_N1}", alpha=0.3)
    sns.scatterplot(data=top_n2, x="val_rmse", y="n_components", ax=axes[1], label=f"Top {TOP_N2}", color="green")
    axes[1].set_xlabel("RMSE_CV"); axes[1].set_ylabel("Number of components")

    sns.scatterplot(data=top_n2_sorted, x="val_rmse", y="num_vars", ax=axes[2], color="purple")
    axes[2].set_xlabel("RMSE_CV"); axes[2].set_ylabel("Number of variables")

    fig.suptitle(f"{trait}", fontsize=16)
    plt.savefig(os.path.join(output_dir, f"ipls_selection_{trait}.png"), dpi=300)
    plt.close()

    # Save top models CSV
    top_n2_sorted.to_csv(os.path.join(output_dir, f"ipls_top_models_{trait}.csv"), index=False, sep="\t")
    print(f"[{trait}] CSV and plot saved.")

# ==== COMBINED CSV FOR ALL TRAITS ====
df_all = pd.concat(all_selected, ignore_index=True)
df_all.to_csv(os.path.join(output_dir, "ipls_top_models_ALL_TRAITS.csv"), index=False, sep="\t")
print(f"Combined CSV for all traits saved.")


## Generate PLS/iPLS models and save

In [7]:
# Required TXT structure for df_parameters_for_modelling
#
# Each row corresponds to one trained model for a specific dataset, trait, and preprocessing method. Columns:
#
# 1. Dataset
#    - Name of the dataset used (e.g., "Kernel")
#
# 2. Trait
#    - Target variable predicted by the model (e.g., "Betasitosterol", "C_16")
#
# 3. Metric
#    - Preprocessing method applied to the spectral data (e.g., "Mean_SG1_W15_P2", "Median_SG1_W15_P2")
#
# 4. Variables
#    - List of selected spectral bands **ordered from 0 to the last band**.
#      - If all bands are used, can be marked as "All"
#      - For iPLS, can also be summarized in interval notation (e.g., "0.95-5-10" = threshold-interval_size-number_of_intervals)
#
# 5. Number of Components
#    - Number of PLS components used in the model (integer)
#
# 6. Train RMSE Mean
#    - Mean RMSE on the training set
#
# 7. Test RMSE Mean
#    - Mean RMSE on the internal cross-validation set
#
# 8. External RMSE Mean
#    - Mean RMSE on the external (hold-out) test set
#
# 9. Overfit_ratio
#    - Ratio of Test RMSE / Train RMSE (values >1 indicate overfitting)
#
# 10. Train R² Mean
#     - Mean coefficient of determination on the training set
#
# 11. Test R² Mean
#     - Mean coefficient of determination on the internal cross-validation set
#
# 12. External R² Mean
#     - Mean coefficient of determination on the external test set
#
# 13. Bands_selected
#     - List of selected spectral bands (for iPLS) or empty if all bands were used
#
# Notes:
# - Each row corresponds to a single model configuration
# - Columns must appear in the order above for scripts to read them correctly

df_parameters_for_modelling=pd.read_csv(r"C:\Users\Pheno\Documents\database_almondcv3\RESULTADOS_PAPER\MODELOS_SELECCIONADOS_PAPER\selected_models.txt", sep='\t')





In [8]:
# Function: ef.train_and_save_pls_models
#
# Description:
# This function trains and saves PLS (Partial Least Squares) models for each row in a configuration DataFrame (`df_config`).
# It can handle standard PLS or interval-PLS (iPLS), using only selected spectral bands if desired.
# For each model, it optionally generates scatter plots, residual plots, X loadings plots, VIP plots, and band influence plots.
#
# Main steps:
# 1. Create a root folder to save all results (`Results_definitive_pls_models`).
# 2. Loop over each row of `df_config`:
#    - Extract trait, preprocessing metric, number of components, and selected bands (for iPLS).
#    - Filter `df_long` by the metric and pivot to wide format (samples x bands).
#    - If `intervalpls=True`, use only the bands specified in `Bands_selected`.
#    - Train the PLS model with the specified number of components.
#    - Save the trained model to disk.
# 3. If `scatterplot=True`:
#    - Perform an external train/test split (Kennard-Stone).
#    - Generate scatter plots of predicted vs true values for training and external sets.
#    - Compute and display RMSE and R² for calibration, cross-validation, and external sets.
#    - Plot residuals (histogram and top 10 individuals).
#    - Plot X loadings per latent variable.
#    - Plot VIP (Variable Importance in Projection) scores.
#    - Plot band influence (PLS coefficients) with positive/negative areas highlighted.
#
# Arguments:
# df_long : pd.DataFrame
#     Long-format spectral data with columns [ID, Trait, Metric, Band, Value].
# df_config : pd.DataFrame
#     Each row contains model parameters: Trait, Metric, Number of Components, Bands_selected, and performance metrics.
# save_path : str
#     Root folder to save models, plots, and results.
# index_column : str, default 'ID'
#     Column that uniquely identifies each sample.
# test_size : float, default 0.2
#     Proportion of data reserved for the external set.
# scatterplot : bool, default True
#     If True, generate plots (scatter, residuals, loadings, VIP, band influence).
# random_state : int, default 10
#     Random seed for reproducibility.
# scaling : bool, default False
#     If True, scale spectral variables (mean=0, variance=1). PLS internally standardizes by default.
# intervalpls : bool, default False
#     If True, train models only using the selected bands in `Bands_selected` column (iPLS).
#
# Notes:
# - The function creates subfolders for each trait.
# - Band indices in `Bands_selected` are relative positions (0 = first band column after ID/trait).
# - All models, plots, and optionally scalers are saved to disk.



ef.train_and_save_pls_models(df_long,df_config=df_parameters_for_modelling, save_path=results_directory, scatterplot=True, index_column="ID",
                              test_size=0.3, scaling=False, intervalpls=True)

✅ Model saved: iPLS_Betasitosterol_Mean_SG1_W15_P2_3comp.pkl
📊 Scatter plot saved: Scatter_Betasitosterol_Mean_SG1_W15_P2_3comp.png
📉 Residuals plots saved: Residuals_Histogram_Betasitosterol_Mean_SG1_W15_P2_3comp.png & Top_10_Residuals_Bars_Betasitosterol_Mean_SG1_W15_P2_3comp.png
📈 X Loadings plot saved: XLoadings_Betasitosterol_Mean_SG1_W15_P2_3comp.png
🌟 VIP plot saved: VIP_Betasitosterol_Mean_SG1_W15_P2_3comp.png
📉 Band influence plot saved: BandInfluence_Betasitosterol_Mean_SG1_W15_P2_3comp.png
✅ Model saved: PLS_C_16_Median_SG1_W15_P2_4comp.pkl
📊 Scatter plot saved: Scatter_C_16_Median_SG1_W15_P2_4comp.png
📉 Residuals plots saved: Residuals_Histogram_C_16_Median_SG1_W15_P2_4comp.png & Top_10_Residuals_Bars_C_16_Median_SG1_W15_P2_4comp.png
📈 X Loadings plot saved: XLoadings_C_16_Median_SG1_W15_P2_4comp.png
🌟 VIP plot saved: VIP_C_16_Median_SG1_W15_P2_4comp.png
📉 Band influence plot saved: BandInfluence_C_16_Median_SG1_W15_P2_4comp.png
✅ Model saved: iPLS_C_18_Mean_SG1_W15_P2_2com

## Deploy the models on data from processed images

In [13]:
# Required columns in df_config for apply_saved_pls_models:

# Trait            : str
#     The name of the trait to predict (e.g., 'Fiber', 'Protein').

# Metric           : str
#     The preprocessing or spectral metric used (e.g., 'Mean_SG1_W15_P2').

# model_path       : str
#     Full path to the saved PLS or iPLS model (.pkl).

# scale_path       : str or None
#     Optional path to a saved scaler to apply to the spectral data before prediction.

# Bands_selected   : list or str (optional, only for iPLS)
#     List of band indices to use for prediction (relative to the spectral columns in df_long). 
#     Can be a string that will be parsed as a list, e.g., '[0, 2, 5, 10]'
# IMPORTANT:
# The spectral columns in X must follow the exact order that was used when training the model.
# - For iPLS models: only the selected bands (Bands_selected), **in the same order as in the original input**.
# Any change in order will lead to incorrect predictions, because the model coefficients are aligned with that order.

df_config_model = pd.read_csv(r"C:\Users\Pheno\Documents\database_almondcv3\RESULTADOS_PAPER\MODELOS_SELECCIONADOS_PAPER\selected_models.txt", sep='\t')

In [16]:
# Function: apply_saved_pls_models
#
# Description:
# This function applies previously trained PLS or iPLS models to new spectral data (`df_long`) and generates predictions.
# It can handle full-spectrum PLS or interval-PLS (iPLS), using only selected bands if specified in `df_config`.
#
# Main steps:
# 1. Loop over each row of `df_config`:
#    - Extract trait, preprocessing metric, model path, optional scaler path, and selected bands (iPLS).
#    - Filter `df_long` by the metric and pivot to wide format (samples x bands).
#    - If `ipls=True`, select only the bands specified in `Bands_selected` (relative indices).
#    - Apply scaler if provided.
#    - Load the saved model.
#    - Predict values for all samples and store results in a DataFrame.
# 2. Merge all trait predictions into a single DataFrame.
# 3. If `save_dir` is provided, save the predictions as a tab-separated TXT file.
#
# Arguments:
# df_long : pd.DataFrame
#     Long-format spectral data with columns [ID, Metric, Band, Value].
# df_config : pd.DataFrame
#     Each row contains model parameters: Trait, Metric, model_path, scale_path, Bands_selected.
# index_column : str, default 'ID'
#     Column that uniquely identifies each sample.
# save_dir : str, optional
#     Folder to save merged predictions.
# ipls : bool, default False
#     If True, select only the bands specified in `Bands_selected` for interval-PLS prediction.
#
# Notes:
# - Bands_selected indices are relative to the spectral columns in df_long (0 = first band).
# - The function prints progress messages including loaded models, applied scalers, and any warnings.
# - Returns a merged DataFrame with all predicted traits.



df_preds = ef.apply_saved_pls_models(
    df_long=df_long,
    df_config=df_config_model,
    index_column='ID',
    ipls=True,
    save_dir=results_directory
)

Betasitosterol iPLS
⚠️ No scale for Betasitosterol (Mean_SG1_W15_P2)
📦 Model loaded for Betasitosterol (Mean_SG1_W15_P2)
C_16 PLS
⚠️ No scale for C_16 (Median_SG1_W15_P2)
📦 Model loaded for C_16 (Median_SG1_W15_P2)
C_18 iPLS
⚠️ No scale for C_18 (Mean_SG1_W15_P2)
📦 Model loaded for C_18 (Mean_SG1_W15_P2)
C_18_1 iPLS
⚠️ No scale for C_18_1 (Median_SG1_W15_P2)
📦 Model loaded for C_18_1 (Median_SG1_W15_P2)
Cholesterol PLS
⚠️ No scale for Cholesterol (Mean_SG1_W15_P2)
📦 Model loaded for Cholesterol (Mean_SG1_W15_P2)
Delta7_Stigmasterol PLS
⚠️ No scale for Delta7_Stigmasterol (Median_SG1_W15_P2)
📦 Model loaded for Delta7_Stigmasterol (Median_SG1_W15_P2)
Fats PLS
⚠️ No scale for Fats (Mean_SG1_W15_P2)
📦 Model loaded for Fats (Mean_SG1_W15_P2)
Fiber iPLS
⚠️ No scale for Fiber (Median_SG1_W15_P2)
📦 Model loaded for Fiber (Median_SG1_W15_P2)
Protein iPLS
⚠️ No scale for Protein (Mean_SG1_W15_P2)
📦 Model loaded for Protein (Mean_SG1_W15_P2)
Stigmasterol iPLS
⚠️ No scale for Stigmasterol (Mean_SG